# Lab 4A: Build a triage service
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 4 · about 30 minutes**

**Objectives**
- Use layered configuration with environment profiles
- Keep prompts in a versioned module
- Wrap every call with timeouts, a token guard, validation, and structured logs
- Prove nothing sensitive is hard-coded

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile appcfg.py
"""Layered configuration: code defaults, then environment profile, then environment variables. Keys never live here."""
import os
DEFAULTS = {"MAX_TOKENS": 800, "TIMEOUT_S": 60, "MAX_RETRIES": 2}
PROFILES = {
    "dev":     {"MODEL": "claude-haiku-4-5-20251001", "LOG_LEVEL": "DEBUG"},
    "staging": {"MODEL": "claude-sonnet-5",           "LOG_LEVEL": "INFO"},
    "prod":    {"MODEL": "claude-sonnet-5",           "LOG_LEVEL": "INFO", "TIMEOUT_S": 30},
}
APP_ENV = os.environ.get("APP_ENV", "dev")
_cfg = {**DEFAULTS, **PROFILES[APP_ENV]}
for k in list(_cfg):                      # environment variables override the profile, e.g. CLAUDE_MODEL
    env_name = "CLAUDE_MODEL" if k == "MODEL" else k
    if os.environ.get(env_name):
        _cfg[k] = type(_cfg[k])(os.environ[env_name])
globals().update(_cfg)

In [ ]:
%%writefile prompts.py
PROMPT_VERSION = "triage-v3"
SYSTEM = ("You triage telecom support tickets. Priorities: P1 = outage affecting many users or critical services; "
          "P2 = one customer fully blocked; P3 = everything else. Return only data matching the JSON schema.")
SCHEMA = {"type": "object", "properties": {
    "category": {"type": "string", "enum": ["billing", "outage", "device", "account", "other"]},
    "priority": {"type": "string", "enum": ["P1", "P2", "P3"]},
    "summary": {"type": "string"}},
  "required": ["category", "priority", "summary"], "additionalProperties": False}
def build_messages(ticket_text, account_id):
    # Untrusted customer text stays in the user turn, clearly delimited
    return [{"role": "user", "content": f"<account>{account_id}</account>\n<ticket>{ticket_text}</ticket>"}]

## Part 1: environment profiles
**Predict:** which model and timeout does each environment get?

In [ ]:
import subprocess, sys
for env in ["dev", "staging", "prod"]:
    out = subprocess.run([sys.executable, "-c", "import appcfg; print(appcfg.APP_ENV, appcfg.MODEL, appcfg.TIMEOUT_S, appcfg.MAX_TOKENS)"],
                         env={**os.environ, "APP_ENV": env}, capture_output=True, text=True).stdout.strip()
    print(out)
os.environ["APP_ENV"] = "dev"
import appcfg, prompts
importlib.reload(appcfg); importlib.reload(prompts)
client = anthropic.Anthropic(max_retries=appcfg.MAX_RETRIES, timeout=appcfg.TIMEOUT_S)
print("\nrunning with:", appcfg.APP_ENV, appcfg.MODEL)

## Part 2: the service call
One function every caller uses: a token budget guard, the pinned model, a schema, validation, and a structured log record.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
log = logging.getLogger("triage")
CALL_LOG = []
MAX_INPUT_TOKENS = 4000

class TriageError(Exception): pass

def validate(d, ticket_text):
    if d["category"] == "outage" and re.search(r"street|sector|hospital|everyone|whole|all customers", ticket_text, re.I) and d["priority"] != "P1":
        raise TriageError("wide outage must be P1")
    return d

def triage(ticket_text, account_id):
    messages = prompts.build_messages(ticket_text, account_id)
    n_in = client.messages.count_tokens(model=appcfg.MODEL, system=prompts.SYSTEM, messages=messages).input_tokens
    if n_in > MAX_INPUT_TOKENS:
        raise TriageError(f"input too large: {n_in} tokens")
    t0 = time.perf_counter()
    resp = client.messages.create(model=appcfg.MODEL, max_tokens=appcfg.MAX_TOKENS, system=prompts.SYSTEM, messages=messages,
                                  output_config={"format": {"type": "json_schema", "schema": prompts.SCHEMA}})
    record = {"request_id": getattr(resp, "_request_id", None), "env": appcfg.APP_ENV, "model": appcfg.MODEL,
              "prompt": prompts.PROMPT_VERSION, "latency_s": round(time.perf_counter() - t0, 2),
              "input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens, "stop_reason": resp.stop_reason}
    try:
        if resp.stop_reason != "end_turn":
            raise TriageError(f"incomplete: stop_reason={resp.stop_reason}")
        result = validate(json.loads("".join(b.text for b in resp.content if b.type == "text")), ticket_text)
        record["outcome"] = "ok"
        return result
    except (TriageError, json.JSONDecodeError) as e:
        record["outcome"] = f"rejected: {e}"
        raise
    finally:
        CALL_LOG.append(record); log.info(json.dumps(record))

In [ ]:
INBOX = [("Whole street in sector 4 has no internet since 6am, a clinic is affected.", "C-2001"),
         ("Charged twice for my September bill.", "C-1002"),
         ("WiFi light blinks orange after a power cut.", "C-1001"),
         ("How do I change my billing email?", "C-1003")]
for text, acct in INBOX:
    try:
        print(triage(text, acct))
    except TriageError as e:
        print("rejected:", e)

In [ ]:
# The call log is your dashboard's raw data
import pandas as pd
pd.DataFrame(CALL_LOG)

## Part 3: the budget guard
**Predict:** what happens with a very long ticket?

In [ ]:
try:
    triage("Router issue. " + "The customer pasted their entire router log here. " * 900, "C-1001")
except TriageError as e:
    print("blocked before calling the model:", e)

## Part 4: nothing sensitive in code
Search the files you wrote for keys and model IDs outside configuration.

In [ ]:
for fname in ["appcfg.py", "prompts.py"]:
    src = open(fname).read()
    print(f"{fname}: contains an API key? {'sk-ant' in src} | model IDs present? {bool(re.search(r'claude-[a-z]', src))}")
print("\nModel IDs should appear only in appcfg.py. The key comes only from the environment.")

## You should now have
- [ ] Profiles for dev, staging, and prod
- [ ] A versioned prompt module
- [ ] A `triage()` call with guard, schema, validation, and logs
- [ ] A call log with request IDs

## Check yourself
1. Why does the client never hold the API key?
2. What should every log record contain?
3. Why log the prompt version?